# Expected value of 17 balance indices represented by the daughter-number basis

This work complements Table 2 in the paper. For small $n$, we can exactly calculate the expected value of a statistic $A(T)$ by the sum $\mathbb{E}[A(T)] = \sum_{T \in \mathcal{T}_n} A(T) \mathbb{P}[T]$, where $\mathbb{P}[T]$ is the probability a random tree with $n$ tips has topology $T$ under the Yule model. However, we can also find the mean as a simple dot product of the mean of the daughter-number basis and the coefficients of that statistic in the daughter-number basis $a_{i,j}$, i.e. $\mathbb{E}[A(T)] = \sum_{i = 1}^{\lfloor n/2 \rfloor} \sum_{j=i}^{n-i} a_{i,j} \mathbb{E}[D_{i,j}(T)]$. Finally, we found simplified formulas using this approach in a Mathematica notebook (found in the folder ~/mathematica), and these simplified expressions can also be used to find the mean. We see that they all agree, and then with the later two methods, we can find the exact values of the statistic means for large $n$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Add the parent directory to Python path so we can import from src
import sys
import os
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

from src.tree_bstats.balance_index_coefficients import BalanceIndexCoefficients
from src.tree_bstats.clade_basis import clade_mean, clade_cov_matrix
from src.tree_bstats.daughter_basis import daughter_mean, daughter_cov_matrix
from src.tree_bstats.tree import Tree
from src.tree_bstats.unlabelled_tree import UnlabelledTree

ModuleNotFoundError: No module named 'numpy'

# Calculate mean of balance indices for small n (up to 18)

We use three methods. The first, stored in array mean, uses the basis representations to calculate the mean. 
The second generates all trees of a given size using the UnlabelledTree class, calculates all the balance indices, and computes the weighted average (with the Yule probability weights) to find the mean of the index under the Yule model.
The last method uses the simplified equations for the mean that we found in Mathematica.  

We observe that these methods generate equivalent results, demonstrating the correctness of the formulas. 

In [ ]:
mean = []
max_n = 18
labels = BalanceIndexCoefficients().get_daughter_labels()

# For all n in the range, we will compute the mean of all basis indices under daughter basis
N = range(4, max_n + 1)
for n in N:
    mu = daughter_mean(n)
    daughter_coef = BalanceIndexCoefficients(n).get_daughter_coefficients()
    matrix_of_coef= np.array(list(daughter_coef.values()))

    mean.append(np.dot(matrix_of_coef, mu))

mean = np.array(mean)

In [ ]:
# Instead, we can also compute the means explicitly using all the trees of a given size
max_n = 18
all_trees = UnlabelledTree(max_n)

mean_all_trees = []

for n in np.arange(4, max_n + 1):
    _, _, balance_indices, weights = all_trees.get_data(n)
    balance_indices = np.array(balance_indices)
    unlabelledTreeLabels = all_trees.balance_index_names
    n_means = np.zeros((len(labels),))
    for i, label in enumerate(labels):
        correct_col = unlabelledTreeLabels.index(label)
        mean_stat = np.sum(balance_indices[:, correct_col] * weights) / np.sum(weights)
        n_means[i] = mean_stat
    mean_all_trees.append(n_means)
mean_all_trees = np.array(mean_all_trees)

   

In [ ]:
# Plot both methods together
for i, label in enumerate(labels):
    plt.plot(N, mean[:, i], label=label + ' (daughter basis)', marker='o')
    plt.plot(N, mean_all_trees[:, i], label=label + ' (all trees)', marker='x')
    plt.xlabel('n')
    plt.ylabel('Expectation under Yule model')
    plt.title(label)
    plt.legend()
    plt.show()

In [ ]:
# Let's see what the differences are between the two methods
diff = np.max(np.abs(mean - mean_all_trees), axis=0)
max_diff_loc = np.argmax(diff)
labels[max_diff_loc], diff[max_diff_loc]

This makes sense, because it has the largest magnitude, so the error from numerical methods will be slightly higher

In [ ]:
# As an alternative, we can compute the means using our formulas from the file mathematica/MeanCalculations.nb

# First, precompute a bunch of harmonic numbers for efficiency
maxN = 2000
harmonic_numbers = np.cumsum( 1 / np.arange(1, maxN + 1))
squared_harmonic_numbers = np.cumsum( 1 / np.arange(1, maxN + 1) ** 2)

def harmonicNumber(n, k=1):
    n = int(n)
    if k == 1 and n <= maxN:
        return harmonic_numbers[n - 1]
    elif k == 2 and n <= maxN:
        return squared_harmonic_numbers[n - 1]
    else:
        return np.sum(1 / np.arange(1, n+1) ** k)

def sackin_index(n):
    return 2 * n * (harmonicNumber(n) - 1)

def average_leaf_depth(n):
    return 2 * (harmonicNumber(n) - 1)

def total_cophenetic_index(n):
    return n * (n + 1 - 2 * harmonicNumber(n))

def s_hat_shape(n):
    return np.log(n - 1) + 2 * n * np.sum( np.log(np.arange(2, n) - 1) / np.arange(2, n) / (np.arange(2, n) + 1))

def colless_index(n):
    if n & 1 == 0:
        return n * (harmonicNumber(n / 2) - 1)
    else:
        return n * (harmonicNumber((n - 1) / 2) - 1) + 1
    
def corrected_colless_index(n):
    return colless_index(n) * 2 / (n-2) / (n-1)

def equal_weights_colless_index(n):
    if n & 1 == 0:
        return n / 6.0 / (n-2) * (4 * (harmonicNumber(n) - harmonicNumber(n / 2)) - 1 / (n - 1) )
    else:
        return (3.0 - n + 4 * n * (n - 2) * (harmonicNumber(n - 1) - harmonicNumber((n - 1) / 2))) / (6.0 * (n - 2)**2)
   
def quadratic_colless_index(n):
    return n * (n + 1 - 2 * harmonicNumber(n))

def total_I_index(n):
    if n & 1 == 0:
        return n * (harmonicNumber(n / 2 - 1) - harmonicNumber(n - 1) + 1)
    else:
        return n * (harmonicNumber((n-1) / 2) - harmonicNumber(n - 1) + 1) - 1

def total_I_prime_index(n):
    if n & 1 == 0:
        return 1.0 / 2 + n * (5.0 / 3 - 2.0 * (harmonicNumber(n-1) - harmonicNumber(n/2 - 1)))
    else:
        return 1.0 / 6 / (n - 1) * (9  + n * ((10 * n - 7) + 12 * (n-1) * (harmonicNumber((n - 3) / 2) - harmonicNumber(n - 2))))
    
def rogers_J_index(n):
    return 2 * total_I_index(n)

def rooted_quartet_binary_index(n):
    return 1 / 72 * n * (n - 1) * (n - 2) * (n - 3)

def total_internal_path_length(n):
    return 2 - 4 * n + 2 * n * harmonicNumber(n)

def total_path_length(n):
    return 2 - 6 * n + 4 * n * harmonicNumber(n)

def average_vertex_depth(n):
    return total_path_length(n) / (2 * n - 1)

def stairs1(n):
    return rogers_J_index(n) / (n - 1)

def g(k):
    if k & 1 == 0:
        return (1 - k - 2 * k * harmonicNumber(k / 2) + 2 * k * harmonicNumber(k)) / ((k - 1))
    else:
        return (1 - k - 2 * k * harmonicNumber((k - 1) / 2) + 2 * k * harmonicNumber(k - 1)) / ((k - 1))
    
def stairs2(n):
    return (1 / (n- 1)) * ((g(n) + 2 * n * sum( g(k) / k / (k + 1) for k in range(2, n))))


In [ ]:
# Now we can compute the means using the formulas
mean_functions = [
    sackin_index,
    average_leaf_depth,
    total_cophenetic_index,
    s_hat_shape,
    colless_index,
    corrected_colless_index,
    equal_weights_colless_index,
    quadratic_colless_index,
    total_I_index,
    total_I_prime_index,
    rogers_J_index,
    rooted_quartet_binary_index,
    total_internal_path_length,
    total_path_length,
    average_vertex_depth,
    stairs1,
    stairs2
]


max_n = 18
N = range(4, max_n + 1)
mean_formula = np.zeros((len(N), len(mean_functions)))
for i, n in enumerate(N):
    for j, func in enumerate(mean_functions):
        mean_formula[i, j] = func(n)


In [ ]:
# Compare the arrays

np.abs(mean_formula - mean_all_trees).max(axis=0)

In [ ]:
# Plot all methods together
for i, label in enumerate(labels):
    plt.plot(N, mean[:, i], label=label + ' (daughter basis)', marker='o')
    plt.plot(N, mean_all_trees[:, i], label=label + ' (all trees)', marker='x')
    plt.plot(N, mean_formula[:, i], label=label + ' (formula)', linestyle='--')
    plt.xlabel('n')
    plt.ylabel('Expectation under Yule model')
    plt.title(label)
    plt.legend()
    plt.show()

# Computing the mean of all indices represented by the daughter-number basis for large n ($n = 100$)

In [ ]:
mean = []
max_n = 100
labels = BalanceIndexCoefficients().get_daughter_labels()

# For all n in the range, we will compute the mean of all basis indices under daughter basis
N = range(4, max_n + 1)
for n in N:
    mu = daughter_mean(n)
    daughter_coef = BalanceIndexCoefficients(n).get_daughter_coefficients()
    matrix_of_coef= np.array(list(daughter_coef.values()))

    mean.append(np.dot(matrix_of_coef, mu))

mean = np.array(mean)

In [ ]:
# Plot the means on a semilogy scale
colors = plt.color_sequences['tab20b']
markers = ['o', '^', 's', '>', 'D', 'x']
linestyles = ['-', '--', '-.']

plt.figure(figsize=(10, 6.2))
for i, label in enumerate(labels):
    plt.semilogy(N, mean[:, i], label=label, 
                 linestyle = linestyles[i % len(linestyles)], 
                 marker = markers[i % len(markers)], markevery=10,
                 color = colors[i % len(colors)])

plt.xlabel('n', fontsize=20)
plt.ylabel('Mean', fontsize=20)
# plt.title('Exact mean of 17 balance indices under Yule model', fontsize=20)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize = 12)
# Fontsize for the x and y axes
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(parent_dir, 'plots', 'meanPlot.png'), dpi=300, bbox_inches='tight')
plt.show()

# Using the formulas we found, we can solve this all for very large n (i.e., n = 2000)

In [ ]:
max_n = 2000
N = range(4, max_n + 1)
mean_formula = np.zeros((len(N), len(mean_functions)))
for i, n in enumerate(N):
    for j, func in enumerate(mean_functions):
        mean_formula[i, j] = func(n)


# Plot the means from the formulas
plt.figure(figsize=(10, 6))
for i, label in enumerate(labels):
    plt.semilogy(N, mean_formula[:, i], label=label)
plt.xlabel('n', fontsize=20)
plt.ylabel('Mean', fontsize=20)
plt.title('Mean of Daughter Basis Indices', fontsize=20)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize = 12)  
plt.xticks(fontsize=12)
plt.yticks(fontsize=16)
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(parent_dir, 'plots', 'meanPlotLargeN.png'), dpi=300, bbox_inches='tight')
plt.show()